# Payments Profiling (API + CSV) — Pre-Gate Checks

This notebook is for exploration and debugging:
- Profile inbound datasets (API mock + CSV)
- Preview likely quality issues before running the strict Great Expectations gate
- Validate canonical selection rule (“CSV wins”)

In [ ]:
import os
import pandas as pd
from sqlalchemy import text

from src.db import create_postgres_engine

engine = create_postgres_engine()
print("Connected to DB:", os.getenv("POSTGRES_DB"))

In [ ]:
df_raw = pd.read_sql(
    "SELECT txn_id, partner_id, status, amount, currency, txn_ts, ingested_at FROM raw_payments ORDER BY txn_id, partner_id;",
    engine,
)
df_raw

In [ ]:
print("Rows:", len(df_raw))
print("\nNull counts:\n", df_raw.isna().sum())

print("\nStatus distribution:\n", df_raw["status"].value_counts(dropna=False))
print("\nCurrency distribution:\n", df_raw["currency"].value_counts(dropna=False))

print("\nAmount stats:\n", df_raw["amount"].describe())

In [ ]:
dupes = (
    df_raw.groupby("txn_id")
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)
dupes.head(20)

In [ ]:
overlap_ids = dupes[dupes["count"] > 1]["txn_id"].tolist()
df_overlap = df_raw[df_raw["txn_id"].isin(overlap_ids)].copy()
df_overlap.sort_values(["txn_id", "partner_id"])

In [ ]:
create_view_sql = """
CREATE OR REPLACE VIEW raw_payments_canonical AS
SELECT DISTINCT ON (txn_id)
  txn_id,
  partner_id,
  status,
  amount,
  currency,
  txn_ts,
  ingested_at
FROM raw_payments
ORDER BY
  txn_id,
  CASE WHEN partner_id = 'partner_csv' THEN 2 ELSE 1 END DESC,
  ingested_at DESC;
"""
with engine.begin() as conn:
    conn.execute(text(create_view_sql))

df_canon = pd.read_sql(
    "SELECT txn_id, partner_id, status, amount, currency, txn_ts, ingested_at FROM raw_payments_canonical ORDER BY txn_id;",
    engine,
)
df_canon

In [ ]:
df_join = df_overlap.merge(
    df_canon[["txn_id", "partner_id", "status"]].rename(columns={"partner_id": "canon_partner", "status": "canon_status"}),
    on="txn_id",
    how="left",
)
df_join.sort_values(["txn_id", "partner_id"])

In [ ]:
# Mimic key GE checks before running the pipeline
issues = {}

issues["negative_amount_rows"] = df_raw[df_raw["amount"] < 0][["txn_id", "partner_id", "amount"]]
issues["bad_currency_rows"] = df_raw[~df_raw["currency"].astype(str).str.match(r"^[A-Z]{3}$", na=False)][["txn_id", "partner_id", "currency"]]

allowed_status = {"AUTHORIZED", "CAPTURED", "REFUNDED", "DECLINED"}
issues["bad_status_rows"] = df_raw[~df_raw["status"].isin(allowed_status)][["txn_id", "partner_id", "status"]]

for k, v in issues.items():
    print("\n==", k, "==")
    display(v.head(50))

## Notes
- This notebook is for exploration only.
- Production logic lives in `src/` and is enforced via Great Expectations + pytest.
- Canonical selection logic is applied through `raw_payments_canonical` (CSV wins, then newest ingested_at).